In [ ]:
import tensorflow as tf
import pathlib, os

url = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"

zip_path = tf.keras.utils.get_file(
    'cats_and_dogs.zip',
    origin=url,
    extract=True
)

data_dir = os.path.join(os.path.dirname(zip_path), 'PetImages')
print(f"📂 Data loaded at: {data_dir}")


824887076/824887076 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step
📂 Data loaded at: /root/.keras/datasets/PetImages


In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers , models
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import tensorflow_datasets as tfds

In [ ]:
data_dir = os.path.join(zip_path, 'PetImages')

In [ ]:
import os
from PIL import Image
import pathlib

num_skipped = 0
for folder_name in ("Cat", "Dog"):
    folder_path = os.path.join(data_dir, folder_name)
    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        try:
            fobj = open(fpath, "rb")
            is_jfif = tf.compat.as_bytes("JFIF") in fobj.peek(10)
        finally:
            fobj.close()

        if not is_jfif:
            num_skipped += 1
            # Delete corrupted image
            os.remove(fpath)

print(f"Deleted {num_skipped} images.")


Deleted 1590 images.


In [ ]:
train_data = tf.keras.utils.image_dataset_from_directory(data_dir , validation_split=0.2 , subset = 'training' , seed=42 , image_size=(150,150) , batch_size=32)


Found 23410 files belonging to 2 classes.
Using 18728 files for training.


In [ ]:
val_data = tf.keras.utils.image_dataset_from_directory(data_dir , validation_split=0.2 , subset = 'validation' , seed=42 , image_size=(150,150) , batch_size=32)


Found 23410 files belonging to 2 classes.
Using 4682 files for validation.


In [ ]:
NL = layers.Rescaling(1./255)

In [ ]:
train_data = train_data.map(lambda x ,y : (NL(x) , y))
val_data = val_data.map(lambda x ,y : (NL(x) , y))

In [ ]:
model = models.Sequential([

     layers.Input(shape=(150, 150, 3)),
     layers.RandomFlip("horizontal") , layers.RandomRotation(0.2) , layers.RandomZoom(0.2),

    layers.Conv2D(32 , (3,3) , activation='relu' , input_shape = (150 , 150 , 3)), layers.MaxPooling2D((2,2)),

    layers.Conv2D(64 , (3,3), activation='relu', input_shape = (150 , 150 , 3)) , layers.MaxPooling2D((2,2)),

    layers.Conv2D(64 , (3,3) , activation='relu' , input_shape = (150 , 150 , 3)) , layers.MaxPooling2D((2,2)),

    layers.Flatten() ,
    layers.Dense(512 , activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1 , activation='sigmoid')
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 150, 150, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18496)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     9,470,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,527,297 (36.34 MB)

 Trainable params: 9,527,297 (36.34 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam' , loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
es = EarlyStopping(monitor='val_loss' , patience =3 , restore_best_weights = True)

In [ ]:
history = model.fit(train_data, epochs=25 , validation_data=val_data , callbacks=[es])

Epoch 1/25
586/586 ━━━━━━━━━━━━━━━━━━━━ 34s 48ms/step - accuracy: 0.5327 - loss: 0.6937 - val_accuracy: 0.5669 - val_loss: 0.6778
Epoch 2/25
586/586 ━━━━━━━━━━━━━━━━━━━━ 37s 47ms/step - accuracy: 0.5807 - loss: 0.6763 - val_accuracy: 0.5886 - val_loss: 0.6607
Epoch 3/25
586/586 ━━━━━━━━━━━━━━━━━━━━ 41s 47ms/step - accuracy: 0.6270 - loss: 0.6471 - val_accuracy: 0.6815 - val_loss: 0.6077
Epoch 4/25
586/586 ━━━━━━━━━━━━━━━━━━━━ 27s 47ms/step - accuracy: 0.6759 - loss: 0.6047 - val_accuracy: 0.7093 - val_loss: 0.5722
Epoch 5/25
586/586 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.7034 - loss: 0.5746 - val_accuracy: 0.7354 - val_loss: 0.5380
Epoch 6/25
586/586 ━━━━━━━━━━━━━━━━━━━━ 27s 46ms/step - accuracy: 0.7226 - loss: 0.5493 - val_accuracy: 0.7627 - val_loss: 0.4867
Epoch 7/25
586/586 ━━━━━━━━━━━━━━━━━━━━ 27s 46ms/step - accuracy: 0.7367 - loss: 0.5306 - val_accuracy: 0.7704 - val_loss: 0.4822
Epoch 8/25
586/586 ━━━━━━━━━━━━━━━━━━━━ 28s 47ms/step - accuracy: 0.7430 - loss: 0.5186 - 

In [ ]:
val_loss , accuracy = model.evaluate(val_data)

print("validation loss = ", val_loss)
print("accuracy = ", accuracy)

147/147 ━━━━━━━━━━━━━━━━━━━━ 6s 39ms/step - accuracy: 0.8477 - loss: 0.3515
validation loss =  0.3514740765094757
accuracy =  0.8477146625518799


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
def predict_image(image_path):
  img = tf.keras.utils.load_img(image_path, target_size=(150,150))
  img_array = tf.keras.utils.img_to_array(img)
  img_array = img_array / 255.0
  img_array = tf.expand_dims(img_array , 0)

  pred = model.predict(img_array , verbose=0)

  if pred[0][0] > 0.5:
        label = "🐶 DOG"
        confidence = pred[0][0]
  else:
        label = "🐱 CAT"
        confidence = 1 - pred[0][0]



  print(f"Prediction: {label} (Confidence: {confidence:.2f})")




In [ ]:
!ls "/content/drive/MyDrive/Colab"

cata.webp	   mycat.jpg   newcat.webp
CATVSDOGCNN.ipynb  mydog.webp  Untitled0.ipynb


In [ ]:
predict_image("/content/drive/MyDrive/Colab/newcat.webp")

Prediction: 🐶 DOG (Confidence: 0.55)


In [ ]:
model.save(r"d:\CNN projects\cat_vs_dog_model.keras")

In [ ]:
img_size = (150 , 150)
batch_size = 32

train_data_tl = tf.keras.utils.image_dataset_from_directory(
    data_dir , validation_split = 0.2 , subset = 'training' , seed = 42 , image_size = img_size , batch_size = batch_size)

val_data_tl = tf.keras.utils.image_dataset_from_directory(
    data_dir , validation_split = 0.2 , subset = 'validation' , seed = 42 , image_size = img_size , batch_size = batch_size)


Found 23410 files belonging to 2 classes.
Using 18728 files for training.
Found 23410 files belonging to 2 classes.
Using 4682 files for validation.


In [ ]:
from tensorflow.keras.applications import VGG16

In [ ]:
base_model = VGG16(

    weights = 'imagenet',
    include_top = False ,
    input_shape = (150,150,3)
)

base_model.trainable = False

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model_tl = models.Sequential([

    layers.Rescaling(1./127.5 , offset = -1 , input_shape = (150, 150 , 3)) ,

    base_model ,
layers.GlobalAveragePooling2D() ,
    layers.Dense(256 , activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1 ,activation = 'sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model_tl.compile(
    optimizer = 'adam' ,
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

In [ ]:
model_tl.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,846,273 (56.63 MB)

 Trainable params: 131,585 (514.00 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(
    monitor = 'val_loss',
    patience = 3 ,
    restore_best_weights = True
)

In [ ]:
htl = model_tl.fit(
    train_data_tl ,
    epochs = 10 ,
    validation_data = val_data_tl ,
    callbacks = [es]
)

Epoch 1/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 99s 169ms/step - accuracy: 0.9123 - loss: 0.2098 - val_accuracy: 0.9370 - val_loss: 0.1467
Epoch 2/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 92s 157ms/step - accuracy: 0.9322 - loss: 0.1632 - val_accuracy: 0.9338 - val_loss: 0.1591
Epoch 3/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 92s 157ms/step - accuracy: 0.9362 - loss: 0.1525 - val_accuracy: 0.9417 - val_loss: 0.1353
Epoch 4/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 89s 152ms/step - accuracy: 0.9399 - loss: 0.1443 - val_accuracy: 0.9425 - val_loss: 0.1301
Epoch 5/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 89s 152ms/step - accuracy: 0.9427 - loss: 0.1383 - val_accuracy: 0.9440 - val_loss: 0.1290
Epoch 6/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 89s 152ms/step - accuracy: 0.9436 - loss: 0.1358 - val_accuracy: 0.9451 - val_loss: 0.1283
Epoch 7/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 89s 152ms/step - accuracy: 0.9444 - loss: 0.1316 - val_accuracy: 0.9445 - val_loss: 0.1301
Epoch 8/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 89s 152ms/step - accuracy: 0.9481 - loss: 0

In [ ]:
val_loss , accuracy = model_tl.evaluate(val_data_tl)

print("validation loss = ", val_loss)
print("accuracy = ", accuracy)

147/147 ━━━━━━━━━━━━━━━━━━━━ 19s 129ms/step - accuracy: 0.9477 - loss: 0.1226
validation loss =  0.12255574017763138
accuracy =  0.9476719498634338


In [ ]:
from tensorflow.keras.applications.vgg16 import preprocess_input

def predict_image_tl(image_path):
  img = tf.keras.utils.load_img(image_path, target_size=(150,150))
  img_array = tf.keras.utils.img_to_array(img)

  img_array = tf.expand_dims(img_array , 0)
  img_array_p = preprocess_input(img_array)

  pred_tl = model_tl.predict(img_array_p , verbose=0)
  if pred_tl[0][0] > 0.5:
      label = "🐶 DOG"
      confidence = pred_tl[0][0]
  else:
      label = "🐱 CAT"
      confidence = 1 - pred_tl[0][0]
  print(f"Prediction: {label} (Confidence: {confidence:.2f})")

In [ ]:
predict_image_tl("/content/drive/MyDrive/Colab/mycat.jpg")

Prediction: 🐱 CAT (Confidence: 1.00)


In [ ]:
model_tl.save(r"d:\CNN projects\expert_cat_dog_vgg16.keras")
